In [0]:
# Databricks Notebook: Transaction_Summary
# Cell 1: Aggregate Transaction Flows by Branch and Type

from pyspark.sql.functions import col, count, current_timestamp, sum, to_date

df_txn = spark.table("bankingpoc.silver.transaction")
df_acc = spark.table("bankingpoc.silver.account")
df_branch = spark.table("bankingpoc.silver.branch")

# Enrich transactions with branch and account metadata
df_enriched_txn = df_txn.join(df_acc, "account_id", "inner") \
                        .join(df_branch, "branch_id", "inner")

df_txn_summary = df_enriched_txn.groupBy(
    to_date("transaction_date").alias("transaction_day"),
    col("branch_name"),
    col("city"),
    col("transaction_type")
).agg(
    count("transaction_id").alias("transaction_count"),
    sum("amount").cast("decimal(18,2)").alias("total_amount")
).withColumn("gold_processed_timestamp", current_timestamp())

(df_txn_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bankingpoc.gold.transaction_summary"))

print("Gold table bankingpoc.gold.transaction_summary successfully generated.")

Gold table bankingpoc.gold.transaction_summary successfully generated.
